# Customer Segmentation

## Project Information
- **Data Source**: Customer Segmentation Dataset (Marketing Analytics)
- **Objective**: Segment customers into distinct groups using K-Means clustering
- **Method**: Unsupervised learning with K-Means clustering
- **Optimal Clusters**: 4 (determined by Elbow method and Silhouette score)

## Dataset
The dataset contains customer information including:
- Demographics (Year of Birth, Education, Marital Status, Income)
- Purchase behavior (amount spent on different product categories)
- Engagement metrics (number of purchases, web visits, campaign responses)
- Target: Identify distinct customer segments for targeted marketing


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)


In [ ]:
# Load and explore the dataset
df = pd.read_csv('customer_segmentation.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.info()
df.head()


## Data Preprocessing


In [ ]:
# Handle missing values
df['Income'] = df['Income'].fillna(df['Income'].median())

# Convert date column
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format="%d-%m-%Y")
df['buy_year'] = df['Dt_Customer'].dt.year.astype(int)
df['buy_month'] = df['Dt_Customer'].dt.month.astype(int)

# Clean marital status (handle unusual values)
df['Marital_Status'] = df['Marital_Status'].replace({'YOLO': 'Single', 'Absurd': 'Single'})

# Drop unnecessary columns
drop_cols = ['ID', 'Dt_Customer', 'Z_CostContact', 'Z_Revenue']
df = df.drop(columns=drop_cols)

# Remove any remaining missing values
df.dropna(inplace=True)

print(f"Dataset shape after preprocessing: {df.shape}")
df.head()


In [ ]:
# Feature engineering: Create aggregated features
df['age'] = df['buy_year'] - df['Year_Birth']

# Total amount spent across all categories
df['total_spent'] = df[[col for col in df.columns if col.startswith('Mnt')]].sum(axis=1)

# Total number of purchases across all channels
df['total_num_purchases'] = df[[col for col in df.columns if col.startswith('Num') and col.endswith('Purchases')]].sum(axis=1)

# Total accepted campaigns
df['total_accepted_camps'] = df[[col for col in df.columns if col.startswith('AcceptedCmp')]].sum(axis=1)

print(f"New features created: age, total_spent, total_num_purchases, total_accepted_camps")
print(f"Final dataset shape: {df.shape}")


## Preprocessing for Clustering


In [ ]:
# Identify categorical and continuous columns
cat_cols = df.select_dtypes(exclude=['float', 'int']).columns.tolist()
cont_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Continuous columns: {len(cont_cols)} features")

# Create preprocessing pipeline
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
scaler = StandardScaler()

preprocess = ColumnTransformer(
    transformers=[
        ('cat', encoder, cat_cols),
        ('cont', scaler, cont_cols)
    ]
)

# Apply preprocessing
processed = preprocess.fit_transform(df)
print(f"\nProcessed data shape: {processed.shape}")


## Finding Optimal Number of Clusters


In [ ]:
# Elbow Method: Find optimal k by minimizing inertia
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(processed)
    inertias.append(km.inertia_)

# Plot elbow curve
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(k_range, inertias, 'r-o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.grid(True)

# Silhouette Score: Measure cluster quality
sil_scores = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(processed)
    s_score = silhouette_score(processed, labels)
    sil_scores.append(s_score)

plt.subplot(1, 2, 2)
plt.plot(k_range, sil_scores, 'k--o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K')
plt.grid(True)

plt.tight_layout()
plt.show()

# Find optimal k (highest silhouette score)
optimal_k = k_range[np.argmax(sil_scores)]
print(f"Optimal number of clusters: {optimal_k} (Silhouette Score: {max(sil_scores):.4f})")


## K-Means Clustering


In [ ]:
# Apply K-Means with optimal k
k = optimal_k
kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
kmeans.fit(processed)

# Assign cluster labels
df['cluster'] = kmeans.labels_.astype(int)

print(f"Clustering complete with {k} clusters")
print(f"Cluster distribution:\n{df['cluster'].value_counts().sort_index()}")


## Cluster Visualization


In [ ]:
# Visualize clusters using key features
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(x='Income', y='total_num_purchases', data=df, hue='cluster', palette='Set2')
plt.title('Customer Segments: Income vs Total Purchases')
plt.xlabel('Income')
plt.ylabel('Total Number of Purchases')

plt.subplot(1, 2, 2)
sns.scatterplot(x='Income', y='total_spent', data=df, hue='cluster', palette='Set2')
plt.title('Customer Segments: Income vs Total Spent')
plt.xlabel('Income')
plt.ylabel('Total Amount Spent')

plt.tight_layout()
plt.show()


## t-SNE Visualization
n

In [ ]:
# Apply t-SNE for 2D visualization of high-dimensional clusters
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
tsne_data = tsne.fit_transform(processed)
tsne_df = pd.DataFrame(tsne_data, columns=['x', 'y'])
tsne_df['cluster'] = df['cluster'].values

# Plot t-SNE visualization
plt.figure(figsize=(10, 8))
sns.scatterplot(x='x', y='y', data=tsne_df, hue='cluster', palette='Set2', s=50, alpha=0.6)
plt.title('t-SNE Visualization of Customer Segments')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.legend(title='Cluster')
plt.show()


## Cluster Analysis


In [ ]:
# Analyze cluster characteristics
cluster_summary = df.groupby('cluster').agg({
    'Income': 'mean',
    'age': 'mean',
    'total_spent': 'mean',
    'total_num_purchases': 'mean',
    'total_accepted_camps': 'mean',
    'Recency': 'mean'
}).round(2)

print("Cluster Characteristics:")
print(cluster_summary)

# Count customers per cluster
print(f"\nCluster Sizes:")
print(df['cluster'].value_counts().sort_index())
